In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/phyothaw/04-market-context-data-ipynb/__results__.html
/kaggle/input/notebooks/phyothaw/04-market-context-data-ipynb/__notebook__.ipynb
/kaggle/input/notebooks/phyothaw/04-market-context-data-ipynb/__output__.json
/kaggle/input/notebooks/phyothaw/04-market-context-data-ipynb/market_context_processed_2020_2025.csv
/kaggle/input/notebooks/phyothaw/04-market-context-data-ipynb/custom.css
/kaggle/input/notebooks/phyothaw/02-5yr-historicdata-ipynb/__results__.html
/kaggle/input/notebooks/phyothaw/02-5yr-historicdata-ipynb/__notebook__.ipynb
/kaggle/input/notebooks/phyothaw/02-5yr-historicdata-ipynb/__output__.json
/kaggle/input/notebooks/phyothaw/02-5yr-historicdata-ipynb/custom.css
/kaggle/input/notebooks/phyothaw/02-5yr-historicdata-ipynb/stock_data_output/all_tickers_combined.csv
/kaggle/input/notebooks/phyothaw/02-5yr-historicdata-ipynb/__results___files/__results___3_3.png
/kaggle/input/notebooks/phyothaw/02-5yr-historicdata-ipynb/__results___files/__results___3

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf

In [3]:
technical_file=("/kaggle/input/notebooks/phyothaw/02-5yr-historicdata-ipynb/stock_data_2020_2025/all_tickers_2020_2025.csv")
gdelt_file=("/kaggle/input/notebooks/phyothaw/03-gdelt-tone-ipynb/gdelt_all_tickers_processed_2020_2025.csv")
market_file=("/kaggle/input/notebooks/phyothaw/04-market-context-data-ipynb/market_context_processed_2020_2025.csv")

technical_df = pd.read_csv(technical_file)
gdelt_df= pd.read_csv(gdelt_file)
market_df= pd.read_csv(market_file)

print("Technical Shape:", technical_df.shape)
print("\nGDELT Shape:", gdelt_df.shape)
print("\nMarket-Context Shape:", market_df.shape)

Technical Shape: (7535, 23)

GDELT Shape: (10960, 10)

Market-Context Shape: (1508, 8)


In [4]:
print("Technical columns:")
print(technical_df.columns.tolist())

print("\nGDELT columns:")
print(gdelt_df.columns.tolist())

print("\nMarket-context columns:")
print(market_df.columns.tolist())

Technical columns:
['Ticker', 'Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits', 'SMA_50', 'SMA_200', 'EMA_50', 'EMA_200', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Upper', 'BB_Middle', 'BB_Lower', 'ATR', 'ADX', 'date_only']

GDELT columns:
['date', 'ticker', 'gdelt_company_tone', 'gdelt_tone_missing', 'gdelt_tone_lag_1', 'gdelt_tone_mean_3', 'gdelt_tone_mean_5', 'gdelt_tone_std_5', 'gdelt_tone_change_1d', 'gdelt_abnormal_tone_20']

Market-context columns:
['date', 'market_close', 'market_return', 'vix_close', 'vix_change', 'market_return_lag_1', 'vix_close_lag_1', 'vix_change_lag_1']


In [5]:
# Standardise the technical dataset column names.
#
# The pre-filtered technical dataset already covers the common
# 2020–2025 study period.
#
# date_only is used as the merge date because it contains a clean
# calendar date without the time and timezone information found in Date.

technical_df = technical_df.rename(columns={"Ticker": "ticker","date_only": "date"})
technical_df = technical_df.drop(columns=["Date"],errors="ignore")

In [6]:
technical_df["date"] = pd.to_datetime(technical_df["date"],errors="coerce")
gdelt_df["date"] = pd.to_datetime(gdelt_df["date"],errors="coerce")
market_df["date"] = pd.to_datetime(market_df["date"],errors="coerce")

#means that if a piece of data cannot be changed to a new data type, the code will quietly change that bad value to a missing value (NaN or NaT) instead of crashing your program.

technical_df["ticker"] = (technical_df["ticker"].astype(str).str.strip().str.upper())
gdelt_df["ticker"] = (gdelt_df["ticker"].astype(str).str.strip().str.upper())

### validation part

In [7]:
print("Technical shape:", technical_df.shape)

print("\nTechnical date range:")
print(
    technical_df["date"].min(),
    "to",
    technical_df["date"].max()
)

print("\nRows per ticker:")
print(
    technical_df["ticker"]
    .value_counts()
    .sort_index()
)

print("\nDuplicate ticker-date rows:")
print(
    technical_df.duplicated(
        subset=["ticker", "date"]
    ).sum()
)

Technical shape: (7535, 22)

Technical date range:
2020-01-02 00:00:00 to 2025-12-30 00:00:00

Rows per ticker:
ticker
BRK-B    1507
CVX      1507
GE       1507
MSFT     1507
NVDA     1507
Name: count, dtype: int64

Duplicate ticker-date rows:
0


In [8]:
print("Technical invalid dates:")
print(technical_df["date"].isna().sum())

print("\nGDELT invalid dates:")
print(gdelt_df["date"].isna().sum())

print("\nMarket invalid dates:")
print(market_df["date"].isna().sum())

print("\nGDELT duplicate ticker-date rows:")
print(
    gdelt_df.duplicated(
        subset=["ticker", "date"]
    ).sum()
)

print("\nMarket duplicate dates:")
print(
    market_df.duplicated(
        subset=["date"]
    ).sum()
)

Technical invalid dates:
0

GDELT invalid dates:
0

Market invalid dates:
0

GDELT duplicate ticker-date rows:
0

Market duplicate dates:
0


In [9]:
# Compare the number of available dates in the technical and GDELT datasets.
#
# The technical dataset contains only stock-market trading days.
# The GDELT dataset contains all calendar days, including weekends and
# market holidays, because media coverage continues when the market is closed.

technical_dates_per_ticker = (technical_df.groupby("ticker")["date"].nunique().reset_index(name="technical_trading_days"))

gdelt_dates_per_ticker = (gdelt_df.groupby("ticker")["date"].nunique().reset_index(name="gdelt_calendar_days"))

date_count_comparison = technical_dates_per_ticker.merge(gdelt_dates_per_ticker,on="ticker",how="inner")

date_count_comparison["non_trading_gdelt_days"] = (
date_count_comparison["gdelt_calendar_days"] - date_count_comparison["technical_trading_days"])

display(date_count_comparison)
print("\nTotal additional GDELT calendar rows:",date_count_comparison["non_trading_gdelt_days"].sum())

,ticker,technical_trading_days,gdelt_calendar_days,non_trading_gdelt_days
0,BRK-B,1507,2192,685
1,CVX,1507,2192,685
2,GE,1507,2192,685
3,MSFT,1507,2192,685
4,NVDA,1507,2192,685



Total additional GDELT calendar rows: 3425


In [10]:
gdelt_df["day_name"] = (gdelt_df["date"].dt.day_name())
gdelt_df["is_weekend"] = (gdelt_df["date"].dt.dayofweek >= 5)
weekend_summary = (gdelt_df.groupby("ticker")["is_weekend"].sum().reset_index(name="weekend_rows"))

display(weekend_summary)

,ticker,weekend_rows
0,BRK-B,626
1,CVX,626
2,GE,626
3,MSFT,626
4,NVDA,626


In [11]:
technical_keys = technical_df[["ticker", "date"]].drop_duplicates()
gdelt_keys = gdelt_df[["ticker", "date"]].drop_duplicates()
gdelt_date_check = gdelt_keys.merge(technical_keys,on=["ticker", "date"],how="left",indicator=True)

non_trading_gdelt_rows = gdelt_date_check[gdelt_date_check["_merge"] == "left_only"].copy()

print("GDELT rows without a matching technical trading date:", len(non_trading_gdelt_rows))

display(non_trading_gdelt_rows.head(20))
# The indicator=True option creates a new column called "_merge":
#
# "both"      -> the ticker-date exists in both GDELT and technical data
# "left_only" -> the ticker-date exists only in GDELT

GDELT rows without a matching technical trading date: 3425


,ticker,date,_merge
0,BRK-B,2020-01-01,left_only
3,BRK-B,2020-01-04,left_only
4,BRK-B,2020-01-05,left_only
10,BRK-B,2020-01-11,left_only
11,BRK-B,2020-01-12,left_only
17,BRK-B,2020-01-18,left_only
18,BRK-B,2020-01-19,left_only
19,BRK-B,2020-01-20,left_only
24,BRK-B,2020-01-25,left_only
25,BRK-B,2020-01-26,left_only


## Verification of Non-Trading GDELT Dates

GDELT provides media-tone observations for all calendar days, while the
technical dataset contains only stock-market trading days.

A left merge is used here only as a validation step. Rows labelled
`left_only` exist in GDELT but not in the technical dataset. These dates
mainly represent weekends and US market holidays.

They are not treated as errors because media coverage continues when the
stock market is closed. These dates were retained when creating lagged and
rolling tone features, but they will not appear as separate rows in the
final trading-day modelling dataset.

In [12]:
# Merge company-level technical indicators with company-specific
# GDELT tone features.
technical_gdelt_df = technical_df.merge(gdelt_df, on=["ticker", "date"], how="left", validate="one_to_one")

print("Technical shape:", technical_df.shape)
print("After GDELT merge:", technical_gdelt_df.shape)
print("Row count unchanged:",len(technical_gdelt_df) == len(technical_df))

display(technical_gdelt_df.head())

Technical shape: (7535, 22)
After GDELT merge: (7535, 32)
Row count unchanged: True


,ticker,Open,High,Low,Close,Volume,Dividends,Stock Splits,SMA_50,SMA_200,...,gdelt_company_tone,gdelt_tone_missing,gdelt_tone_lag_1,gdelt_tone_mean_3,gdelt_tone_mean_5,gdelt_tone_std_5,gdelt_tone_change_1d,gdelt_abnormal_tone_20,day_name,is_weekend
0,NVDA,5.934968,5.963804,5.884506,5.963804,237536000,0.0,0.0,5.355496,4.491415,...,2.2550,0,1.0359,1.035900,1.03590,NaN,NaN,NaN,Thursday,False
1,NVDA,5.844236,5.912100,5.819378,5.868349,205384000,0.0,0.0,5.375684,4.499142,...,1.4056,0,2.2550,1.645450,1.64545,0.862034,1.2191,NaN,Friday,False
2,NVDA,5.775127,5.898177,5.749026,5.892956,262636000,0.0,0.0,5.396621,4.505810,...,1.3464,0,1.8434,1.663767,1.65644,0.460595,0.1011,NaN,Monday,False
3,NVDA,5.921294,6.010039,5.876301,5.964300,314856000,0.0,0.0,5.418107,4.513633,...,1.4840,0,1.3464,1.644033,1.71854,0.367547,-0.4970,NaN,Tuesday,False
4,NVDA,5.960074,6.016751,5.920052,5.975486,277108000,0.0,0.0,5.436000,4.521973,...,1.0930,0,1.4840,1.557933,1.56434,0.217195,0.1376,NaN,Wednesday,False


In [13]:
gdelt_unmatched_rows = (technical_gdelt_df["gdelt_tone_missing"].isna().sum())

print("Trading rows with no matching GDELT date:",gdelt_unmatched_rows)

gdelt_columns = [
    "gdelt_company_tone",
    "gdelt_tone_missing",
    "gdelt_tone_lag_1",
    "gdelt_tone_mean_3",
    "gdelt_tone_mean_5",
    "gdelt_tone_std_5",
    "gdelt_tone_change_1d",
    "gdelt_abnormal_tone_20"
]

print("Missing values in GDELT features:")
print(technical_gdelt_df[gdelt_columns].isna().sum())

Trading rows with no matching GDELT date: 0
Missing values in GDELT features:
gdelt_company_tone         65
gdelt_tone_missing          0
gdelt_tone_lag_1           70
gdelt_tone_mean_3          50
gdelt_tone_mean_5          45
gdelt_tone_std_5           55
gdelt_tone_change_1d       85
gdelt_abnormal_tone_20    135
dtype: int64


## GDELT Feature Meaning and Missing-Value Check

- `gdelt_company_tone`: raw daily media tone for the company.
- `gdelt_tone_missing`: indicates whether the raw tone was unavailable.
- `gdelt_tone_lag_1`: tone from the previous calendar day.
- `gdelt_tone_mean_3`: average tone over the previous 3 calendar days.
- `gdelt_tone_mean_5`: average tone over the previous 5 calendar days.
- `gdelt_tone_std_5`: variation in tone over the previous 5 calendar days.
- `gdelt_tone_change_1d`: change between the previous two daily tone values.
- `gdelt_abnormal_tone_20`: recent tone relative to its previous 20-day average.

All technical trading dates matched the GDELT dataset. The remaining missing
values are expected because some original GDELT dates were unavailable and
lagged or rolling features require earlier observations.

In [14]:
# Save the technical and GDELT merged dataset before adding
# S&P 500 and VIX market-context variables.

technical_gdelt_output = ("/kaggle/working/technical_gdelt_merged_2020_2025.csv")

technical_gdelt_df.to_csv(
    technical_gdelt_output,
    index=False)

print("Saved:", technical_gdelt_output)
print("Shape:", technical_gdelt_df.shape)

Saved: /kaggle/working/technical_gdelt_merged_2020_2025.csv
Shape: (7535, 32)


### **Merge market context**

In [15]:
# The market-context dataset has one row per date, while the combined technical-GDELT dataset has multiple ticker rows for the same date.
#
# validate="many_to_one" confirms that many stock rows can match one
# market-context row for each trading date.

merged_df = technical_gdelt_df.merge(market_df,on="date",how="left",validate="many_to_one")

print("Before market merge:", technical_gdelt_df.shape)
print("After market merge:", merged_df.shape)
print("Final row count unchanged:",len(merged_df) == len(technical_df))

display(merged_df.head())

Before market merge: (7535, 32)
After market merge: (7535, 39)
Final row count unchanged: True


,ticker,Open,High,Low,Close,Volume,Dividends,Stock Splits,SMA_50,SMA_200,...,gdelt_abnormal_tone_20,day_name,is_weekend,market_close,market_return,vix_close,vix_change,market_return_lag_1,vix_close_lag_1,vix_change_lag_1
0,NVDA,5.934968,5.963804,5.884506,5.963804,237536000,0.0,0.0,5.355496,4.491415,...,NaN,Thursday,False,3257.850098,NaN,12.47,NaN,NaN,NaN,NaN
1,NVDA,5.844236,5.912100,5.819378,5.868349,205384000,0.0,0.0,5.375684,4.499142,...,NaN,Friday,False,3234.850098,-0.007060,14.02,0.124298,NaN,12.47,NaN
2,NVDA,5.775127,5.898177,5.749026,5.892956,262636000,0.0,0.0,5.396621,4.505810,...,NaN,Monday,False,3246.280029,0.003533,13.85,-0.012126,-0.007060,14.02,0.124298
3,NVDA,5.921294,6.010039,5.876301,5.964300,314856000,0.0,0.0,5.418107,4.513633,...,NaN,Tuesday,False,3237.179932,-0.002803,13.79,-0.004332,0.003533,13.85,-0.012126
4,NVDA,5.960074,6.016751,5.920052,5.975486,277108000,0.0,0.0,5.436000,4.521973,...,NaN,Wednesday,False,3253.050049,0.004902,13.45,-0.024656,-0.002803,13.79,-0.004332


In [16]:
market_unmatched_rows = (merged_df["market_close"].isna().sum())

print("Trading rows with no matching market-context date:",market_unmatched_rows)

market_columns = [
    "market_close",
    "market_return",
    "vix_close",
    "vix_change",
    "market_return_lag_1",
    "vix_close_lag_1",
    "vix_change_lag_1"]

print("Missing values in market-context features:")
print(merged_df[market_columns].isna().sum())

Trading rows with no matching market-context date: 0
Missing values in market-context features:
market_close            0
market_return           5
vix_close               0
vix_change              5
market_return_lag_1    10
vix_close_lag_1         5
vix_change_lag_1       10
dtype: int64


In [17]:
print("Market dataframe columns:")
print(market_df.columns.tolist())

print("\nMissing values before merging:")
print(market_df.isna().sum())

display(
    market_df[
        [
            "date",
            "market_close",
            "vix_close",
            "vix_change",
            "vix_close_lag_1",
            "vix_change_lag_1"
        ]
    ].head(10)
)

Market dataframe columns:
['date', 'market_close', 'market_return', 'vix_close', 'vix_change', 'market_return_lag_1', 'vix_close_lag_1', 'vix_change_lag_1']

Missing values before merging:
date                   0
market_close           0
market_return          1
vix_close              0
vix_change             1
market_return_lag_1    2
vix_close_lag_1        1
vix_change_lag_1       2
dtype: int64


,date,market_close,vix_close,vix_change,vix_close_lag_1,vix_change_lag_1
0,2020-01-02,3257.850098,12.47,NaN,NaN,NaN
1,2020-01-03,3234.850098,14.02,0.124298,12.47,NaN
2,2020-01-06,3246.280029,13.85,-0.012126,14.02,0.124298
3,2020-01-07,3237.179932,13.79,-0.004332,13.85,-0.012126
4,2020-01-08,3253.050049,13.45,-0.024656,13.79,-0.004332
5,2020-01-09,3274.699951,12.54,-0.067658,13.45,-0.024656
6,2020-01-10,3265.350098,12.56,0.001595,12.54,-0.067658
7,2020-01-13,3288.129883,12.32,-0.019108,12.56,0.001595
8,2020-01-14,3283.149902,12.39,0.005682,12.32,-0.019108
9,2020-01-15,3289.290039,12.42,0.002421,12.39,0.005682


In [18]:
merged_df = merged_df.drop(columns=["day_name","is_weekend"],errors="ignore")
print("Final shape:", merged_df.shape)

print("\nDuplicate ticker-date rows:")
print(merged_df.duplicated(subset=["ticker", "date"]).sum())

print("\nMarket-context missing values:")
print(
    merged_df[[
            "market_close",
            "market_return",
            "vix_close",
            "vix_change",
            "market_return_lag_1",
            "vix_close_lag_1",
            "vix_change_lag_1"]].isna().sum())

Final shape: (7535, 37)

Duplicate ticker-date rows:
0

Market-context missing values:
market_close            0
market_return           5
vix_close               0
vix_change              5
market_return_lag_1    10
vix_close_lag_1         5
vix_change_lag_1       10
dtype: int64


In [19]:
# Save the fully merged dataset containing technical indicators,
# GDELT tone features, and market-context variables.

merged_output_file = ("/kaggle/working/""technical_gdelt_market_merged_2020_2025.csv")

merged_df.to_csv(merged_output_file,index=False)

print("Saved:", merged_output_file)
print("Final shape:", merged_df.shape)

Saved: /kaggle/working/technical_gdelt_market_merged_2020_2025.csv
Final shape: (7535, 37)


In [20]:
merged_check = pd.read_csv(
    merged_output_file
)

print("Loaded shape:", merged_check.shape)

print("\nDuplicate ticker-date rows:")
print(
    merged_check.duplicated(
        subset=["ticker", "date"]
    ).sum()
)

display(merged_check.head())

Loaded shape: (7535, 37)

Duplicate ticker-date rows:
0


,ticker,Open,High,Low,Close,Volume,Dividends,Stock Splits,SMA_50,SMA_200,...,gdelt_tone_std_5,gdelt_tone_change_1d,gdelt_abnormal_tone_20,market_close,market_return,vix_close,vix_change,market_return_lag_1,vix_close_lag_1,vix_change_lag_1
0,NVDA,5.934968,5.963804,5.884506,5.963804,237536000,0.0,0.0,5.355496,4.491415,...,NaN,NaN,NaN,3257.850098,NaN,12.47,NaN,NaN,NaN,NaN
1,NVDA,5.844236,5.912100,5.819378,5.868349,205384000,0.0,0.0,5.375684,4.499142,...,0.862034,1.2191,NaN,3234.850098,-0.007060,14.02,0.124298,NaN,12.47,NaN
2,NVDA,5.775127,5.898177,5.749026,5.892956,262636000,0.0,0.0,5.396621,4.505810,...,0.460595,0.1011,NaN,3246.280029,0.003533,13.85,-0.012126,-0.007060,14.02,0.124298
3,NVDA,5.921294,6.010039,5.876301,5.964300,314856000,0.0,0.0,5.418107,4.513633,...,0.367547,-0.4970,NaN,3237.179932,-0.002803,13.79,-0.004332,0.003533,13.85,-0.012126
4,NVDA,5.960074,6.016751,5.920052,5.975486,277108000,0.0,0.0,5.436000,4.521973,...,0.217195,0.1376,NaN,3253.050049,0.004902,13.45,-0.024656,-0.002803,13.79,-0.004332
